# Ensemble TWAS Weights via Stacked Regression

## Overview

pecotmr offers 10+ TWAS weight methods (SuSiE, LASSO, Elastic Net, mr.ash, BayesR/L/A/C, SCAD, MCP, L0Learn), each suited to different genetic architectures. Picking one method is arbitrary; running all and testing each incurs a multiple-testing penalty. **Stacked regression** (SR-TWAS; Dai et al., *Nature Communications* 2024, [doi:10.1038/s41467-024-50983-w](https://doi.org/10.1038/s41467-024-50983-w)) provides a principled alternative: learn one convex combination of methods per gene, producing a single weight vector for downstream TWAS testing with no method-selection bias.

This notebook:
1. Presents the mathematical framework and algorithm for ensemble weight learning
2. Benchmarks the approach via simulation across independent replicates
3. Produces diagnostic plots comparing CV $R^2$, MSE, and ensemble coefficients

| Step | Description | Parallelism |
|------|-------------|-------------|
| `simulate_and_fit` | Simulate X and Y per replicate, run 11 methods + ensemble, save RDS | Per-replicate (`for_each`) |
| `analyze_results` | Load RDS files, compute metrics, generate diagnostic plots | Single task |

## Mathematical Framework

### Gene Expression Model

For a target gene $g$ with $p$ cis-SNPs, the expression level of individual $i$ is:

$$E_i = \mathbf{G}_i \mathbf{w} + \epsilon_i, \quad \epsilon_i \sim N(0, \sigma^2_\epsilon)$$

where $\mathbf{G}_i \in \mathbb{R}^{1 \times p}$ is the genotype vector and $\mathbf{w} \in \mathbb{R}^p$ is the true cis-eQTL effect-size vector.

### Base Models

We train $K$ prediction models $\{m_1, \ldots, m_K\}$, each producing a weight estimate $\hat{\mathbf{w}}_k \in \mathbb{R}^p$. Under $F$-fold cross-validation, the out-of-fold prediction for sample $i$ from method $k$ is:

$$\hat{E}_i^{(k)} = \mathbf{G}_i \hat{\mathbf{w}}_k^{(-f_i)}$$

where $f_i$ denotes the fold containing sample $i$ and $\hat{\mathbf{w}}_k^{(-f)}$ is trained on all samples except fold $f$. Stacking all $n$ out-of-fold predictions across $K$ methods yields the prediction matrix:

$$\mathbf{P} = \begin{pmatrix} \hat{E}_1^{(1)} & \cdots & \hat{E}_1^{(K)} \\ \vdots & \ddots & \vdots \\ \hat{E}_n^{(1)} & \cdots & \hat{E}_n^{(K)} \end{pmatrix} \in \mathbb{R}^{n \times K}$$

### Stacked Regression (Constrained QP)

We seek ensemble coefficients $\boldsymbol{\zeta} = (\zeta_1, \ldots, \zeta_K)^\top$ that minimize the prediction error of the convex combination:

$$\min_{\boldsymbol{\zeta}} \;\; \left\| \mathbf{E} - \sum_{k=1}^{K} \zeta_k \hat{\mathbf{E}}^{(k)} \right\|^2 = \left\| \mathbf{E} - \mathbf{P} \boldsymbol{\zeta} \right\|^2$$

$$\text{subject to} \quad \zeta_k \geq 0 \;\; \forall k, \qquad \sum_{k=1}^{K} \zeta_k = 1$$

This is a convex quadratic program solved via `quadprog::solve.QP` with:

$$\mathbf{D} = \mathbf{P}^\top \mathbf{P} + \lambda \mathbf{I}, \quad \mathbf{d} = \mathbf{P}^\top \mathbf{E}$$

where $\lambda = 10^{-8} \cdot \text{mean}(\text{diag}(\mathbf{P}^\top \mathbf{P}))$ provides ridge regularization for numerical stability.

### Ensemble Weight Vector

The final ensemble TWAS weight vector combines the full-data weight estimates:

$$\tilde{\mathbf{w}} = \sum_{k=1}^{K} \zeta_k^* \hat{\mathbf{w}}_k$$

### Performance Metrics

- **CV $R^2$**: $R^2_{\text{CV}} = \text{cor}(\mathbf{E},\; \hat{\mathbf{E}}_{\text{ens}})^2$ where $\hat{\mathbf{E}}_{\text{ens}} = \mathbf{P} \boldsymbol{\zeta}^*$
- **Prediction MSE**: $\text{MSE} = \frac{1}{n} \| \mathbf{G}\hat{\mathbf{w}} - \mathbf{G}\mathbf{w}_{\text{true}} \|^2$ (against known truth in simulation)

## Algorithm

The algorithm below shows the existing pecotmr cross-validation framework (Steps 1--5) and the **new ensemble learning extension** (Steps 6--9, in bold). Step numbers reference the mathematical formulation above.

```
Algorithm: TWAS Weight Estimation with Ensemble Learning
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Input:  Genotype matrix G (n x p), expression vector E (n x 1),
        weight methods M = {m_1, ..., m_K}, number of CV folds F
Output: Per-method weights {w_k}, ensemble coefficients {zeta_k},
        combined weight vector w_tilde, CV performance metrics

── Existing Framework: twas_weights_cv() + twas_weights() ──────

1. Partition n samples into F folds: {S_1, ..., S_F}
2. For each fold f = 1, ..., F:
   For each method k = 1, ..., K:
     a. Train:   w_k^(-f)  <-  m_k(G[T_f,], E[T_f])
     b. Predict:  E_hat^(k)[S_f]  <-  G[S_f,] * w_k^(-f)
3. Assemble out-of-fold prediction matrix P  (n x K)
     P[i,k] = E_hat_i^(k) from the fold containing sample i
4. Compute per-method performance:
     R^2_k = cor(E, P[,k])^2    for k = 1, ..., K
5. Train final weights on full data:
     w_k  <-  m_k(G, E)    for all k

── NEW: Ensemble Learning via ensemble_weights() ──────────────

  6. Solve constrained QP for ensemble coefficients:
       D  <-  P^T P + lambda * I      (ridge regularization)
       d  <-  P^T E
       zeta*  <-  argmin  1/2 zeta^T D zeta - d^T zeta
                  s.t.  sum(zeta_k) = 1,  zeta_k >= 0  for all k
  7. Compute ensemble prediction:
       E_hat_ens  =  P zeta*
  8. Compute ensemble performance:
       R^2_ens  =  cor(E, E_hat_ens)^2
  9. Combine final weight vectors:
       w_tilde  =  sum_k  zeta_k * w_k

Return: {w_k}, {zeta_k}, w_tilde, {R^2_k}, R^2_ens
```

**Implementation**: Steps 1--5 are `pecotmr::twas_weights_cv()` and `pecotmr::twas_weights()`. Steps 6--9 are `ensemble_weights()` from `R/ensemble_weights.R`.

## Weight Methods

| # | Method | Type | pecotmr function | Key parameters |
|---|--------|------|------------------|----------------|
| 1 | SuSiE | Bayesian sparse | `susie_weights` | `refine=FALSE, init_L=5, max_L=10` |
| 2 | LASSO | Penalized regression | `lasso_weights` | default (`glmnet`) |
| 3 | Elastic Net | Penalized regression | `enet_weights` | default (`glmnet`, `alpha=0.5`) |
| 4 | mr.ash | Bayesian adaptive shrinkage | `mrash_weights` | `init_prior_sd=TRUE` |
| 5 | BayesR | Bayesian mixture | `bayes_r_weights` | default (`qgg`) |
| 6 | BayesL | Bayesian LASSO | `bayes_l_weights` | default (`qgg`) |
| 7 | BayesA | Bayesian ridge-like | `bayes_a_weights` | default (`qgg`) |
| 8 | BayesC | Bayesian spike-slab | `bayes_c_weights` | default (`qgg`) |
| 9 | SCAD | Nonconvex penalized | `scad_weights` | default (`ncvreg`) |
| 10 | MCP | Nonconvex penalized | `mcp_weights` | default (`ncvreg`) |
| 11 | L0Learn | Best-subset approx | `l0learn_weights` | default (`L0Learn`) |

## Simulation Design

### Genotypes

Each replicate simulates independent genotypes via `simxQTL::sim_geno_indep(n, p)` with MAF in $[0.05, 0.4]$. We use simulated (not real) genotypes so that the true effect sizes and causal architecture are known exactly for MSE evaluation.

### Phenotypes

Gene expression is generated via `simxQTL::generate_cis_qtl_data()` with a mixed genetic architecture:
- **Sparse**: `n_sparse` variants with large effects
- **Oligogenic**: `n_oligogenic` variants with moderate effects
- **Infinitesimal**: `n_inf` variants with small effects
- Total heritability: $h^2_g$ (default 0.2)

### Replicates

Two modes are supported for defining replicates:

- **Region-based** (default): An LD metadata file defines genomic regions (e.g., LD blocks). Each region provides a replicate identifier and can be crossed with `n_seeds` independent seeds. For example, 20 regions with `n_seeds = 10` yields 200 replicates. Use `--chrom` to restrict to a specific chromosome or `--chrom 0` for all.

- **Count-based**: Set `--n-replicates` directly to run a fixed number of replicates without a region file. Each replicate is identified by its index.

### Ground Truth

Since `generate_cis_qtl_data()` returns the causal variant indices and true effect sizes, we compute:
- **Prediction MSE**: $\frac{1}{n} \| \mathbf{G}\hat{\mathbf{w}} - \mathbf{G}\mathbf{w}_{\text{true}} \|^2$
- **CV $R^2$**: correlation-based, from out-of-fold predictions

## Usage

### Quick test (10 replicates)

```bash
sos run analysis/ensemble_twas_weights.ipynb simulate_and_fit \
    --n-replicates 10 \
    --n-samples 500 \
    --n-variants 500 \
    --h2g 0.2 \
    -J 10
```

### Region-based (using LD metadata file)

```bash
sos run analysis/ensemble_twas_weights.ipynb simulate_and_fit \
    --ld-meta-file /path/to/ld_meta_file.tsv \
    --chrom 22 \
    --n-seeds 10 \
    -J 40
```

### Full benchmark (200 replicates)

```bash
sos run analysis/ensemble_twas_weights.ipynb simulate_and_fit \
    --n-replicates 200 \
    -J 40
```

### Analysis and plots

```bash
sos run analysis/ensemble_twas_weights.ipynb analyze_results
```

# Pipeline Implementation

In [ ]:
[global]
# Output directory
parameter: cwd = path("output/ensemble_twas")

# Path to ensemble_weights.R source file
parameter: ensemble_weights_source = path("R/ensemble_weights.R")

# ── Replicate definition ─────────────────────────────────────
# Option A: Fixed replicate count (set > 0 to use)
parameter: n_replicates = 0

# Option B: Region-based from LD metadata file
parameter: ld_meta_file = path("")
parameter: chrom = 0
parameter: n_seeds = 1

# Simulation parameters
parameter: n_samples = 500
parameter: n_variants = 500
parameter: h2g = 0.2
parameter: n_sparse = 3
parameter: n_oligogenic = 5
parameter: n_inf = 15
parameter: cv_folds = 5
parameter: max_cv_variants = 500

# Random seed
parameter: seed_base = 42

# Cluster resources
parameter: job_size = 1
parameter: walltime = "4:00:00"
parameter: mem = "16G"
parameter: numThreads = 4

import os
cwd = path(f'{cwd:a}')

def _read_ld_regions(meta_file, chrom_filter):
    """Parse LD metadata TSV for the specified chromosome.
    Returns list of dicts with keys: id, chrom, start, end, region.
    """
    regions = []
    with open(meta_file) as fh:
        for line in fh:
            if line.startswith('#chrom') or line.startswith('chrom'):
                continue
            parts = line.strip().split('\t')
            if len(parts) < 3:
                continue
            c, start, end = parts[0], parts[1], parts[2]
            if not c.startswith('chr'):
                c = f'chr{c}'
            cnum = int(c.replace('chr', ''))
            if chrom_filter != 0 and cnum != chrom_filter:
                continue
            regions.append({
                'id': f'{c}_{start}_{end}',
                'chrom': c,
                'start': int(start),
                'end': int(end),
                'region': f'{c}:{start}-{end}'
            })
    return regions

# Build replicate list
replicates = []
if n_replicates > 0:
    # Count-based mode: simple indexed replicates
    for i in range(n_replicates):
        replicates.append({
            'rep_id': f'rep_{i:04d}',
            'seed': seed_base + i,
            'region': f'sim_{i:04d}'
        })
elif str(ld_meta_file) and os.path.isfile(str(ld_meta_file)):
    # Region-based mode: LD metadata file x seeds
    _base_regions = _read_ld_regions(str(ld_meta_file), chrom)
    for seed_idx in range(n_seeds):
        for i, r in enumerate(_base_regions):
            rep = dict(r)
            rep['seed'] = seed_base + seed_idx * len(_base_regions) + i
            rep['rep_id'] = f'{r["id"]}_seed{seed_idx}'
            replicates.append(rep)
else:
    # Default: 10 replicates
    for i in range(10):
        replicates.append({
            'rep_id': f'rep_{i:04d}',
            'seed': seed_base + i,
            'region': f'sim_{i:04d}'
        })

## Simulate and Fit

For each replicate:
1. Simulate genotype matrix $\mathbf{G}$ via `simxQTL::sim_geno_indep()`
2. Simulate expression $\mathbf{E}$ via `simxQTL::generate_cis_qtl_data()` with mixed architecture
3. Run `pecotmr::twas_weights_pipeline()` with 11 methods and 5-fold CV
4. Call `ensemble_weights()` to learn $\boldsymbol{\zeta}$ and combine weights
5. Save results + ground truth as RDS

In [ ]:
[simulate_and_fit]
input: for_each = "replicates"
output: f'{cwd}/{_replicates["rep_id"]}/results.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads
R: expand = "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout'

    library(pecotmr)
    library(simxQTL)

    # Source the ensemble_weights function
    source("${ensemble_weights_source}")

    # Set seed for reproducibility
    set.seed(${_replicates['seed']})
    cat("Replicate: ${_replicates['rep_id']}\n")
    cat("Seed: ${_replicates['seed']}\n")

    # ── 1. Simulate genotype matrix ──────────────────────────────
    G <- sim_geno_indep(n = ${n_samples}, p = ${n_variants},
                        min_maf = 0.05, max_maf = 0.4)
    X <- scale(G)
    n <- nrow(X); p <- ncol(X)
    cat(sprintf("Genotype matrix: %d samples x %d variants\n", n, p))

    # ── 2. Simulate phenotype with mixed architecture ────────────
    sim <- generate_cis_qtl_data(G, h2g = ${h2g},
                                  n_sparse = ${n_sparse},
                                  n_oligogenic = ${n_oligogenic},
                                  n_inf = ${n_inf})
    y <- sim$y
    cat(sprintf("Realized h2g: %.4f\n", sim$h2g))

    # ── 3. Define 11 weight methods ──────────────────────────────
    weight_methods <- list(
        susie_weights   = list(refine = FALSE, init_L = 5, max_L = 10),
        lasso_weights   = list(),
        enet_weights    = list(),
        mrash_weights   = list(init_prior_sd = TRUE, max.iter = 100),
        bayes_r_weights = list(),
        bayes_l_weights = list(),
        bayes_a_weights = list(),
        bayes_c_weights = list(),
        scad_weights    = list(),
        mcp_weights     = list(),
        l0learn_weights = list()
    )

    # ── 4. Run TWAS weights pipeline (CV + full-data weights) ────
    cat("Running twas_weights_pipeline with", length(weight_methods), "methods...\n")
    res <- tryCatch(
        twas_weights_pipeline(
            X, y,
            cv_folds = ${cv_folds},
            weight_methods = weight_methods,
            max_cv_variants = ${max_cv_variants},
            num_threads = 1
        ),
        error = function(e) {
            cat("Pipeline error:", conditionMessage(e), "\n")
            NULL
        }
    )

    if (is.null(res)) {
        cat("Pipeline failed, saving NULL result\n")
        saveRDS(list(results = NULL, ensemble = NULL,
                     truth = list(rep_id = "${_replicates['rep_id']}",
                                  seed = ${_replicates['seed']},
                                  error = TRUE)),
                "${_output}")
        quit(save = "no", status = 0)
    }

    # ── 5. Ensemble learning (Steps 6-9 in Algorithm) ────────────
    cat("Computing ensemble weights...\n")
    ens <- tryCatch(
        ensemble_weights(
            cv_results = res$twas_cv_result,
            Y = y,
            twas_weight_list = res$twas_weights
        ),
        error = function(e) {
            cat("Ensemble error:", conditionMessage(e), "\n")
            NULL
        }
    )

    # ── 6. Compute MSE against truth ─────────────────────────────
    # True genetic component
    true_beta <- rep(0, p)
    if (!is.null(sim$sparse_indices))       true_beta[sim$sparse_indices] <- sim$sparse_effects
    if (!is.null(sim$oligogenic_indices))    true_beta[sim$oligogenic_indices] <- sim$oligogenic_effects
    if (!is.null(sim$infinitesimal_indices)) true_beta[sim$infinitesimal_indices] <- sim$infinitesimal_effects
    true_genetic <- as.vector(X %*% true_beta)

    # Per-method MSE
    method_mse <- sapply(names(res$twas_weights), function(wname) {
        w_hat <- res$twas_weights[[wname]]
        if (is.matrix(w_hat)) w_hat <- w_hat[, 1]
        pred_genetic <- as.vector(X %*% w_hat)
        mean((pred_genetic - true_genetic)^2)
    })
    names(method_mse) <- gsub("_weights$", "", names(method_mse))

    # Ensemble MSE
    ensemble_mse <- NA
    if (!is.null(ens) && !is.null(ens$ensemble_twas_weights)) {
        w_ens <- ens$ensemble_twas_weights
        if (is.matrix(w_ens)) w_ens <- w_ens[, 1]
        ensemble_mse <- mean((as.vector(X %*% w_ens) - true_genetic)^2)
    }

    # ── 7. Save complete results ─────────────────────────────────
    truth <- list(
        causal_sparse = sim$sparse_indices,
        causal_oligo = sim$oligogenic_indices,
        causal_inf = sim$infinitesimal_indices,
        true_beta = true_beta,
        true_y = y,
        h2g = sim$h2g,
        rep_id = "${_replicates['rep_id']}",
        seed = ${_replicates['seed']},
        n = n, p = p,
        method_mse = method_mse,
        ensemble_mse = ensemble_mse
    )

    out <- list(results = res, ensemble = ens, truth = truth)
    dir.create(dirname("${_output}"), recursive = TRUE, showWarnings = FALSE)
    saveRDS(out, "${_output}")
    cat("Saved:", "${_output}", "\n")
    cat("Ensemble R^2:", if (!is.null(ens)) round(ens$ensemble_performance["rsq"], 4) else NA, "\n")
    cat("Ensemble MSE:", round(ensemble_mse, 6), "\n")

## Analyze Results

Load all per-replicate RDS files and produce:
1. **CV $R^2$ boxplot** — all 11 methods + ensemble, ordered by median
2. **MSE boxplot** — prediction MSE against true genetic component
3. **Coefficient heatmap** — ensemble $\zeta_k$ across replicates
4. **Scatter plot** — ensemble $R^2$ vs. best single-method $R^2$
5. **Summary CSV** — one row per replicate with all metrics

In [ ]:
[analyze_results]
parameter: results_dir = path(f'{cwd}')
output: f'{results_dir}/diagnostics.pdf', f'{results_dir}/summary_table.csv'
task: trunk_workers = 1, walltime = "1:00:00", mem = "8G", cores = 1
R: expand = "${ }", stderr = f'{results_dir}/analyze.stderr', stdout = f'{results_dir}/analyze.stdout'

    library(ggplot2)
    library(tidyr)
    library(dplyr)

    results_dir <- "${results_dir}"

    # ── 1. Load all RDS files ────────────────────────────────────
    rds_files <- list.files(results_dir, pattern = "results\\.rds$",
                            recursive = TRUE, full.names = TRUE)
    cat(sprintf("Found %d RDS files\n", length(rds_files)))

    all_data <- lapply(rds_files, function(f) {
        tryCatch(readRDS(f), error = function(e) NULL)
    })
    all_data <- all_data[!sapply(all_data, is.null)]
    # Remove failed runs
    all_data <- all_data[!sapply(all_data, function(x) isTRUE(x$truth$error))]
    cat(sprintf("%d successful replicates\n", length(all_data)))

    if (length(all_data) == 0) stop("No valid results found")

    # ── 2. Extract metrics ───────────────────────────────────────
    metrics_list <- lapply(seq_along(all_data), function(i) {
        x <- all_data[[i]]
        rep_id <- x$truth$rep_id
        seed <- x$truth$seed

        # Per-method CV R^2 from ensemble output
        method_rsq <- if (!is.null(x$ensemble)) x$ensemble$method_performance else NULL
        # Ensemble CV R^2
        ens_rsq <- if (!is.null(x$ensemble)) x$ensemble$ensemble_performance["rsq"] else NA
        # Ensemble coefficients
        coefs <- if (!is.null(x$ensemble)) x$ensemble$method_coef else NULL
        # MSE
        method_mse <- x$truth$method_mse
        ens_mse <- x$truth$ensemble_mse

        list(
            rep_id = rep_id, seed = seed,
            method_rsq = method_rsq, ensemble_rsq = ens_rsq,
            method_coef = coefs,
            method_mse = method_mse, ensemble_mse = ens_mse
        )
    })

    # ── 3. Build tidy data frames ────────────────────────────────

    # R^2 data frame (long format)
    rsq_rows <- do.call(rbind, lapply(metrics_list, function(m) {
        if (is.null(m$method_rsq)) return(NULL)
        method_df <- data.frame(
            rep_id = m$rep_id,
            method = names(m$method_rsq),
            rsq = as.numeric(m$method_rsq),
            stringsAsFactors = FALSE
        )
        ens_df <- data.frame(
            rep_id = m$rep_id,
            method = "ensemble",
            rsq = as.numeric(m$ensemble_rsq),
            stringsAsFactors = FALSE
        )
        rbind(method_df, ens_df)
    }))

    # MSE data frame (long format)
    mse_rows <- do.call(rbind, lapply(metrics_list, function(m) {
        if (is.null(m$method_mse)) return(NULL)
        method_df <- data.frame(
            rep_id = m$rep_id,
            method = names(m$method_mse),
            mse = as.numeric(m$method_mse),
            stringsAsFactors = FALSE
        )
        ens_df <- data.frame(
            rep_id = m$rep_id,
            method = "ensemble",
            mse = as.numeric(m$ensemble_mse),
            stringsAsFactors = FALSE
        )
        rbind(method_df, ens_df)
    }))

    # Coefficient matrix for heatmap
    coef_mat <- do.call(rbind, lapply(metrics_list, function(m) {
        if (is.null(m$method_coef)) return(NULL)
        m$method_coef
    }))
    if (!is.null(coef_mat)) {
        rownames(coef_mat) <- sapply(metrics_list[!sapply(metrics_list,
            function(m) is.null(m$method_coef))], function(m) m$rep_id)
    }

    # Scatter data: ensemble vs best single method
    scatter_df <- do.call(rbind, lapply(metrics_list, function(m) {
        if (is.null(m$method_rsq)) return(NULL)
        data.frame(
            rep_id = m$rep_id,
            best_single_rsq = max(m$method_rsq, na.rm = TRUE),
            best_method = names(which.max(m$method_rsq)),
            ensemble_rsq = as.numeric(m$ensemble_rsq),
            stringsAsFactors = FALSE
        )
    }))

    # ── 4. Diagnostic plots ──────────────────────────────────────
    pdf("${_output[0]}", width = 12, height = 8)

    # Plot 1: CV R^2 boxplot
    if (!is.null(rsq_rows) && nrow(rsq_rows) > 0) {
        # Order methods by median R^2
        method_order <- rsq_rows %>%
            group_by(method) %>%
            summarise(med = median(rsq, na.rm = TRUE)) %>%
            arrange(med) %>%
            pull(method)
        rsq_rows$method <- factor(rsq_rows$method, levels = method_order)

        p1 <- ggplot(rsq_rows, aes(x = method, y = rsq,
                                    fill = ifelse(method == "ensemble", "Ensemble", "Single"))) +
            geom_boxplot(outlier.size = 0.8) +
            scale_fill_manual(values = c("Ensemble" = "#E41A1C", "Single" = "#377EB8"),
                              name = "") +
            labs(title = "Cross-Validation R-squared by Method",
                 x = "Method", y = expression(CV~R^2)) +
            theme_bw(base_size = 12) +
            theme(axis.text.x = element_text(angle = 45, hjust = 1))
        print(p1)
    }

    # Plot 2: MSE boxplot
    if (!is.null(mse_rows) && nrow(mse_rows) > 0) {
        method_order_mse <- mse_rows %>%
            group_by(method) %>%
            summarise(med = median(mse, na.rm = TRUE)) %>%
            arrange(desc(med)) %>%
            pull(method)
        mse_rows$method <- factor(mse_rows$method, levels = method_order_mse)

        p2 <- ggplot(mse_rows, aes(x = method, y = mse,
                                    fill = ifelse(method == "ensemble", "Ensemble", "Single"))) +
            geom_boxplot(outlier.size = 0.8) +
            scale_fill_manual(values = c("Ensemble" = "#E41A1C", "Single" = "#377EB8"),
                              name = "") +
            labs(title = "Prediction MSE Against True Genetic Component",
                 x = "Method", y = "MSE") +
            theme_bw(base_size = 12) +
            theme(axis.text.x = element_text(angle = 45, hjust = 1))
        print(p2)
    }

    # Plot 3: Coefficient heatmap
    if (!is.null(coef_mat) && nrow(coef_mat) > 1) {
        coef_long <- as.data.frame(coef_mat) %>%
            mutate(replicate = rownames(coef_mat)) %>%
            pivot_longer(-replicate, names_to = "method", values_to = "zeta")

        p3 <- ggplot(coef_long, aes(x = method, y = replicate, fill = zeta)) +
            geom_tile() +
            scale_fill_gradient(low = "white", high = "#E41A1C",
                                name = expression(zeta[k]),
                                limits = c(0, 1)) +
            labs(title = "Ensemble Coefficients Across Replicates",
                 x = "Method", y = "Replicate") +
            theme_bw(base_size = 10) +
            theme(axis.text.x = element_text(angle = 45, hjust = 1),
                  axis.text.y = element_text(size = 6))
        print(p3)
    }

    # Plot 4: Ensemble vs best single-method scatter
    if (!is.null(scatter_df) && nrow(scatter_df) > 0) {
        p4 <- ggplot(scatter_df, aes(x = best_single_rsq, y = ensemble_rsq)) +
            geom_point(alpha = 0.6, size = 2, color = "#377EB8") +
            geom_abline(intercept = 0, slope = 1, linetype = "dashed", color = "grey40") +
            labs(title = "Ensemble R-squared vs Best Single Method",
                 x = expression(Best~single~method~R^2),
                 y = expression(Ensemble~R^2)) +
            theme_bw(base_size = 12) +
            coord_equal()
        print(p4)
    }

    dev.off()
    cat("Saved plots:", "${_output[0]}", "\n")

    # ── 5. Summary table ─────────────────────────────────────────
    summary_rows <- lapply(metrics_list, function(m) {
        row <- data.frame(
            rep_id = m$rep_id,
            seed = m$seed,
            ensemble_rsq = as.numeric(m$ensemble_rsq),
            best_method_rsq = if (!is.null(m$method_rsq)) max(m$method_rsq, na.rm = TRUE) else NA,
            best_method = if (!is.null(m$method_rsq)) names(which.max(m$method_rsq)) else NA,
            ensemble_mse = as.numeric(m$ensemble_mse),
            stringsAsFactors = FALSE
        )
        # Add individual method R^2 and coefficients
        if (!is.null(m$method_rsq)) {
            for (mn in names(m$method_rsq)) {
                row[[paste0(mn, "_rsq")]] <- m$method_rsq[[mn]]
            }
        }
        if (!is.null(m$method_mse)) {
            for (mn in names(m$method_mse)) {
                row[[paste0(mn, "_mse")]] <- m$method_mse[[mn]]
            }
        }
        if (!is.null(m$method_coef)) {
            for (mn in names(m$method_coef)) {
                row[[paste0("zeta_", mn)]] <- m$method_coef[[mn]]
            }
        }
        row
    })
    summary_df <- bind_rows(summary_rows)
    # Compute improvement column
    summary_df$rsq_improvement <- summary_df$ensemble_rsq - summary_df$best_method_rsq

    write.csv(summary_df, "${_output[1]}", row.names = FALSE)
    cat("Saved summary:", "${_output[1]}", "\n")

    # Print summary statistics
    cat("\n=== Summary ===", "\n")
    cat(sprintf("Replicates: %d\n", nrow(summary_df)))
    cat(sprintf("Mean ensemble R^2: %.4f\n", mean(summary_df$ensemble_rsq, na.rm = TRUE)))
    cat(sprintf("Mean best-single R^2: %.4f\n", mean(summary_df$best_method_rsq, na.rm = TRUE)))
    cat(sprintf("Ensemble >= best in %.1f%% of replicates\n",
        100 * mean(summary_df$rsq_improvement >= 0, na.rm = TRUE)))
    cat(sprintf("Mean R^2 improvement: %.4f\n", mean(summary_df$rsq_improvement, na.rm = TRUE)))
    cat(sprintf("Mean ensemble MSE: %.6f\n", mean(summary_df$ensemble_mse, na.rm = TRUE)))

## Expected Results

With default parameters ($n = 500$, $p = 500$, $h^2_g = 0.2$, mixed architecture):

- **CV $R^2$**: Expect values in the range 0.05--0.25 for individual methods; ensemble should match or slightly exceed the best single method.
- **Coefficient patterns**: Sparse architectures favor SuSiE/LASSO (high $\zeta_{\text{susie}}$, $\zeta_{\text{lasso}}$); polygenic architectures favor BayesR/BayesL; the ensemble adapts automatically.
- **MSE**: Ensemble MSE should be competitive with or lower than the best single-method MSE.
- **Scatter plot**: Most points should lie on or above the diagonal, indicating the ensemble matches or improves on the best single method.